In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

logins = pd.read_csv("../../data/processed/login_logs_clean.csv")
access = pd.read_csv("../../data/processed/access_records_clean.csv")

logins["timestamp"] = pd.to_datetime(logins["timestamp"])
print(logins.shape)
logins.head()

(20164, 10)


,log_id,timestamp,employee_id,ip_address,country,device,login_status,session_duration,data_accessed,hour
0,LOG000001,2026-04-01 10:17:00,EMP0153,53.211.247.91,Canada,Mobile,Success,252,Medium,10
1,LOG000002,2026-04-01 13:49:00,EMP0085,219.60.117.228,India,Desktop,Success,265,Medium,13
2,LOG000003,2026-04-01 09:35:00,EMP0212,14.82.199.2,India,Laptop,Success,246,High,9
3,LOG000004,2026-04-01 06:16:00,EMP0119,10.196.205.146,USA,Laptop,Success,424,Low,6
4,LOG000005,2026-04-01 11:00:00,EMP0173,206.146.99.17,USA,Desktop,Success,350,Low,11


In [2]:
# Time-based features
logins["hour"] = logins["timestamp"].dt.hour
logins["is_after_hours"] = logins["hour"].apply(lambda h: 1 if h < 6 or h >= 22 else 0)

# Login outcome
logins["is_failed"] = (logins["login_status"] == "Failed").astype(int)

# Unusual country flag
common_countries = ["USA", "UK", "India", "Singapore", "Germany", "Canada"]
logins["is_unusual_country"] = (~logins["country"].isin(common_countries)).astype(int)

# Unusual data access level, encoded as a number
data_access_map = {"Low": 0, "Medium": 1, "High": 2}
logins["data_accessed_encoded"] = logins["data_accessed"].map(data_access_map).fillna(0)

# How many failed logins did this employee have on this same day? (catches brute-force bursts)
logins["date"] = logins["timestamp"].dt.date
fail_counts = logins[logins["is_failed"] == 1].groupby(["employee_id", "date"]).size()
fail_counts = fail_counts.rename("failed_attempts_same_day")
logins = logins.merge(fail_counts, on=["employee_id", "date"], how="left")
logins["failed_attempts_same_day"] = logins["failed_attempts_same_day"].fillna(0)

# Bring in privilege level from access_records (highest privilege each employee holds)
priv_map = {"Standard": 0, "Elevated": 1, "Admin": 2}
access["privilege_encoded"] = access["privilege_level"].map(priv_map)
max_priv = access.groupby("employee_id")["privilege_encoded"].max().rename("privilege_encoded")
logins = logins.merge(max_priv, on="employee_id", how="left")
logins["privilege_encoded"] = logins["privilege_encoded"].fillna(0)

logins[["hour","is_after_hours","is_failed","is_unusual_country","data_accessed_encoded",
        "failed_attempts_same_day","privilege_encoded","session_duration"]].head()

,hour,is_after_hours,is_failed,is_unusual_country,data_accessed_encoded,failed_attempts_same_day,privilege_encoded,session_duration
0,10,0,0,0,1.0,0.0,2,252
1,13,0,0,0,1.0,0.0,0,265
2,9,0,0,0,2.0,0.0,0,246
3,6,0,0,0,0.0,0.0,0,424
4,11,0,0,0,0.0,0.0,0,350


In [3]:
feature_cols = ["hour", "is_after_hours", "is_failed", "is_unusual_country",
                 "data_accessed_encoded", "failed_attempts_same_day",
                 "privilege_encoded", "session_duration"]

X = logins[feature_cols]

# contamination = our rough guess of what % of logins are anomalous (~5%)
model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
model.fit(X)

logins["anomaly_score"] = model.decision_function(X)          # higher = more normal
logins["anomaly_flag"] = model.predict(X)                     # -1 = anomaly, 1 = normal
logins["anomaly_flag"] = logins["anomaly_flag"].map({-1: "Anomaly", 1: "Normal"})

logins["anomaly_flag"].value_counts()

anomaly_flag
Normal     19155
Anomaly     1009
Name: count, dtype: int64

In [4]:
flagged = logins[logins["anomaly_flag"] == "Anomaly"]
print("Total flagged:", len(flagged))
print("\n% of flagged that are after-hours:", (flagged["is_after_hours"].mean() * 100).round(1))
print("% of flagged from unusual countries:", (flagged["is_unusual_country"].mean() * 100).round(1))
print("% of flagged with failed-login bursts:", (flagged["failed_attempts_same_day"] > 3).mean() * 100)

flagged.sort_values("anomaly_score").head(10)[["employee_id","timestamp","country","is_after_hours","failed_attempts_same_day","anomaly_score"]]

Total flagged: 1009

% of flagged that are after-hours: 57.5
% of flagged from unusual countries: 38.4
% of flagged with failed-login bursts: 2.180376610505451


,employee_id,timestamp,country,is_after_hours,failed_attempts_same_day,anomaly_score
20143,EMP0248,2026-05-25 04:10:00,Canada,1,10.0,-0.117147
20150,EMP0248,2026-05-25 04:34:00,USA,1,10.0,-0.117147
20151,EMP0248,2026-05-25 04:55:00,UK,1,10.0,-0.117147
20145,EMP0248,2026-05-25 04:33:00,UK,1,10.0,-0.117147
20144,EMP0248,2026-05-25 04:18:00,Singapore,1,10.0,-0.117147
20146,EMP0248,2026-05-25 04:43:00,Germany,1,10.0,-0.117147
20147,EMP0248,2026-05-25 04:08:00,Singapore,1,10.0,-0.117147
20152,EMP0248,2026-05-25 04:01:00,Singapore,1,10.0,-0.117147
20149,EMP0248,2026-05-25 04:23:00,Singapore,1,10.0,-0.117147
20148,EMP0248,2026-05-25 04:06:00,Germany,1,10.0,-0.117147


In [5]:
import os, joblib

os.makedirs("../../app/ml/models", exist_ok=True)
joblib.dump(model, "../../app/ml/models/isolation_forest.pkl")

logins.to_csv("../../data/processed/login_logs_with_anomalies.csv", index=False)
print("Saved model and results.")

Saved model and results.
